# Automated Executive Report Generation Pipeline

**Project 04 -- Executive KPI Report** | Notebook 5 of 5

## Purpose

This notebook walks through the **end-to-end pipeline** that transforms raw SaaS metrics into a polished, multi-section executive PDF report -- automatically. The pipeline combines:

1. **Data loading and KPI computation** (from notebooks 01--02)
2. **Anomaly detection** (from notebook 03)
3. **Forecasting** (from notebook 04)
4. **Natural language generation** via template-based commentary (`commentary.py`)
5. **Chart rendering** via plotly -> PNG export
6. **PDF assembly** via fpdf2 (`report_generator.py`)

### Why Automate?

Manual executive reports consume 4--8 hours per cycle for most data teams. This pipeline reduces that to **under 30 seconds**, while ensuring consistency, reproducibility, and zero copy-paste errors.

### Architecture

```
monthly_metrics.parquet ──┐
                          ├──> KPI Computation ──> Anomaly Detection ──> Forecasting
segment_metrics.parquet ──┘         │                    │                    │
                                    v                    v                    v
                            commentary.py ──────> report_generator.py ──────> PDF
                            (NLG templates)       (fpdf2 assembly)       (byte stream)
```

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add backend to path so we can import the report modules
sys.path.insert(0, os.path.abspath("../backend"))

from kpi_backend.analytics_engine import (
    detect_anomalies_zscore,
    detect_anomalies_iqr,
    exponential_smoothing_forecast,
    compute_health_score,
    traffic_light_status,
    mrr_waterfall,
)
from kpi_backend.commentary import (
    generate_executive_summary,
    generate_revenue_commentary,
    generate_customer_commentary,
    generate_anomaly_narrative,
)
from kpi_backend.report_generator import generate_report

# Color palette
COLORS = {
    "cyan": "#06B6D4", "violet": "#8B5CF6", "emerald": "#10B981",
    "amber": "#F59E0B", "rose": "#F43F5E", "navy": "#0F172A",
}

# Load data
monthly = pd.read_parquet("../data/processed/monthly_metrics.parquet")
monthly_kpis = pd.read_parquet("../data/processed/monthly_kpis.parquet")
segments = pd.read_parquet("../data/processed/segment_metrics.parquet")

monthly = monthly.sort_values("month").reset_index(drop=True)
monthly_kpis = monthly_kpis.sort_values("month").reset_index(drop=True)

print(f"Data loaded: {monthly.shape[0]} months, {segments.shape[0]} segment-months")
print(f"Latest month: {monthly['month'].max().strftime('%B %Y')}")

## 1. Step 1: Build the Metrics Payload

The report generator expects a structured `data` dict with specific keys. This step extracts the latest month's metrics and computes derived values (health score, traffic lights, MoM/YoY changes).

This mirrors what the FastAPI `/report/generate` endpoint does in production (`backend/kpi_backend/routers/report.py`).

In [ ]:
# Extract latest month metrics
last_row = monthly.iloc[-1]
prev_row = monthly.iloc[-2]

# Build the metrics dict expected by commentary functions
metrics = {}
for col in monthly.columns:
    if col == "month":
        continue
    try:
        metrics[col] = float(last_row[col])
    except (TypeError, ValueError):
        pass

# Compute MoM growth rate for MRR
metrics["mrr_growth_rate"] = (metrics["mrr"] - float(prev_row["mrr"])) / float(prev_row["mrr"])

# Compute health score using the analytics engine
health = compute_health_score(metrics)
metrics["health_score"] = health

# Build MRR waterfall
waterfall = mrr_waterfall(last_row)

# Period info
period = {
    "start": monthly["month"].min().strftime("%Y-%m"),
    "end": monthly["month"].max().strftime("%Y-%m"),
}

print("Metrics payload built for:", monthly["month"].max().strftime("%B %Y"))
print(f"  MRR:          ${metrics['mrr']:>12,.0f}")
print(f"  ARR:          ${metrics['arr']:>12,.0f}")
print(f"  MRR Growth:   {metrics['mrr_growth_rate']:>11.1%}")
print(f"  NRR:          {metrics['nrr']:>11.1%}")
print(f"  Logo Churn:   {metrics['logo_churn_rate']:>11.1%}")
print(f"  NPS:          {metrics['nps']:>11.0f}")
print(f"  Health Score: {health:>11.1f} / 100")
print(f"\nWaterfall:")
for k, v in waterfall.items():
    print(f"  {k:20s}: ${v:>12,.0f}")

## 2. Step 2: Natural Language Generation (Commentary)

The `commentary.py` module uses **template-based NLG** -- f-string templates with conditional logic that adapt narrative tone based on metric values. This is not an LLM-generated summary; it is deterministic and auditable.

### Why template-based NLG over LLM?

| Aspect | Template NLG | LLM-based |
|--------|-------------|-----------|
| Latency | <1ms | 2-10 seconds |
| Cost | $0 | $0.01-0.10 per report |
| Determinism | 100% reproducible | Varies per run |
| Auditability | Full control over every word | Black-box |
| Bilingual | Explicit ES/EN templates | Prompt engineering |

For executive reports where every word matters and legal/compliance teams review output, template NLG is the safer choice.

In [ ]:
# Generate all commentary sections -- English
exec_summary_en = generate_executive_summary(metrics, lang="en")
revenue_commentary_en = generate_revenue_commentary(metrics, lang="en")
customer_commentary_en = generate_customer_commentary(metrics, lang="en")

print("EXECUTIVE SUMMARY (EN)")
print("=" * 70)
print(exec_summary_en)

print("\n\nREVENUE COMMENTARY (EN)")
print("=" * 70)
print(revenue_commentary_en)

print("\n\nCUSTOMER HEALTH (EN)")
print("=" * 70)
print(customer_commentary_en)

# Generate Spanish versions to demonstrate bilingual capability
exec_summary_es = generate_executive_summary(metrics, lang="es")
print("\n\nRESUMEN EJECUTIVO (ES)")
print("=" * 70)
print(exec_summary_es)

## 3. Step 3: Anomaly Detection for Alerts

The report includes an anomaly alerts section that automatically flags unusual KPI values. We run z-score detection across key metrics and generate a narrative summary.

In [ ]:
# Run anomaly detection across key metrics
month_labels = monthly["month"].apply(lambda m: m.strftime("%Y-%m"))
anomaly_metrics = ["mrr", "logo_churn_rate", "nps", "nrr", "cac"]

all_anomalies = []
for metric in anomaly_metrics:
    anoms = detect_anomalies_zscore(
        monthly[metric].astype(float),
        index_labels=month_labels,
        threshold=2.0,
    )
    for a in anoms:
        a["metric"] = metric
    all_anomalies.extend(anoms)

print(f"Detected {len(all_anomalies)} anomalies across {len(anomaly_metrics)} metrics:")
print("-" * 60)
for a in all_anomalies:
    print(f"  {a['metric']:20s}  {a['month']}  z={a['zscore']:+.2f}  ({a['severity']})")

# Generate anomaly narrative
anomaly_narrative = generate_anomaly_narrative(all_anomalies, lang="en")
print(f"\nANOMALY NARRATIVE:")
print("=" * 70)
print(anomaly_narrative)

## 4. Step 4: Forecasting for the Report

The report includes a forecast section with 6-month projections for MRR, churn, and NPS. The `exponential_smoothing_forecast()` function from `analytics_engine.py` handles model fitting and confidence interval computation.

In [ ]:
# Generate forecasts for the report
forecast_metrics = ["mrr", "logo_churn_rate", "nps"]
forecast_data = {}

for metric in forecast_metrics:
    result = exponential_smoothing_forecast(
        monthly[metric].astype(float),
        periods=6,
    )
    forecast_data[metric] = result

# Display forecast results
future_dates = pd.date_range(
    start=monthly["month"].max() + pd.DateOffset(months=1),
    periods=6, freq="MS",
)

print("6-Month Forecasts")
print("=" * 80)
for metric in forecast_metrics:
    fcast = forecast_data[metric]
    print(f"\n  {metric.upper().replace('_', ' ')}:")
    for i, (date, val, lo, hi) in enumerate(zip(
        future_dates, fcast["forecast"], fcast["ci_lower"], fcast["ci_upper"]
    )):
        if metric == "mrr":
            print(f"    {date.strftime('%b %Y'):>10s}:  ${val:>12,.0f}   (95% CI: ${lo:>12,.0f} -- ${hi:>12,.0f})")
        elif "rate" in metric:
            print(f"    {date.strftime('%b %Y'):>10s}:  {val:>12.4f}   (95% CI: {lo:>12.4f} -- {hi:>12.4f})")
        else:
            print(f"    {date.strftime('%b %Y'):>10s}:  {val:>12.1f}   (95% CI: {lo:>12.1f} -- {hi:>12.1f})")

## 5. Step 5: Chart Export (Plotly -> Static Images)

Executive PDFs need static images, not interactive HTML. The pipeline renders plotly figures to PNG via `fig.write_image()` (using the kaleido engine when available, or falling back to orca).

The `report_generator.py` handles this via `_render_chart_png()`, which writes to a temp file and embeds it in the PDF.

Let's demonstrate chart creation for the report -- the MRR waterfall and MRR trend with forecast.

In [ ]:
# Chart 1: MRR Waterfall for the latest month
fig_waterfall = go.Figure(go.Waterfall(
    x=["Starting MRR", "New MRR", "Expansion", "Contraction", "Churned", "Ending MRR"],
    y=[
        waterfall["starting_mrr"],
        waterfall["new"],
        waterfall["expansion"],
        -waterfall["contraction"],
        -waterfall["churned"],
        waterfall["ending_mrr"],
    ],
    measure=["absolute", "relative", "relative", "relative", "relative", "total"],
    connector=dict(line=dict(color="#d1d5db")),
    increasing=dict(marker=dict(color=COLORS["emerald"])),
    decreasing=dict(marker=dict(color=COLORS["rose"])),
    totals=dict(marker=dict(color=COLORS["cyan"])),
    text=[f"${v:,.0f}" for v in [
        waterfall["starting_mrr"], waterfall["new"], waterfall["expansion"],
        -waterfall["contraction"], -waterfall["churned"], waterfall["ending_mrr"],
    ]],
    textposition="outside",
))

fig_waterfall.update_layout(
    title=f"MRR Waterfall -- {monthly['month'].max().strftime('%B %Y')}",
    yaxis_title="MRR ($)", yaxis_tickprefix="$", yaxis_tickformat=",",
    template="plotly_white", height=400,
    font=dict(family="Inter, sans-serif"),
)
fig_waterfall.show()

# Chart 2: MRR trend with forecast
fcast = forecast_data["mrr"]
fig_trend = go.Figure()

fig_trend.add_trace(go.Scatter(
    x=monthly["month"], y=monthly["mrr"],
    mode="lines+markers", name="Actual MRR",
    line=dict(color=COLORS["cyan"], width=2), marker=dict(size=4),
))
fig_trend.add_trace(go.Scatter(
    x=future_dates, y=fcast["forecast"],
    mode="lines+markers", name="Forecast",
    line=dict(color=COLORS["violet"], width=2),
    marker=dict(size=7, symbol="diamond"),
))
fig_trend.add_trace(go.Scatter(
    x=list(future_dates) + list(future_dates[::-1]),
    y=fcast["ci_upper"] + fcast["ci_lower"][::-1],
    fill="toself", fillcolor="rgba(139, 92, 246, 0.12)",
    line=dict(color="rgba(0,0,0,0)"), name="95% CI",
))
# Convert Timestamp to string for plotly compatibility
forecast_start = monthly["month"].max().strftime("%Y-%m-%d")
fig_trend.add_vline(x=forecast_start, line_dash="dash", line_color="gray")

fig_trend.update_layout(
    title="MRR: Actual and 6-Month Forecast",
    xaxis_title="Month", yaxis_title="MRR ($)",
    yaxis_tickprefix="$", yaxis_tickformat=",",
    template="plotly_white", height=400,
    font=dict(family="Inter, sans-serif"),
    legend=dict(x=0.02, y=0.98),
)
fig_trend.show()

## 6. Step 6: PDF Assembly with fpdf2

The `report_generator.py` module uses **fpdf2** (a pure-Python PDF library) to assemble the final document. The `_KPIReport` class extends `FPDF` with custom header/footer, section titles, body text, and KPI tables.

### PDF Sections

The report contains 8 sections, each independently toggleable:

| Section | Content | Data Source |
|---------|---------|-------------|
| `cover` | Title page with company name and period | Period dates |
| `executive_summary` | 3-5 sentence NLG summary | `commentary.py` |
| `kpi_dashboard` | Color-coded KPI table with traffic lights | All metrics + targets |
| `revenue_analysis` | Revenue narrative + MRR waterfall chart | `commentary.py` + chart |
| `customer_health` | Customer metrics narrative | `commentary.py` |
| `anomaly_alerts` | Flagged anomalies table + narrative | `analytics_engine.py` |
| `forecast` | 6-month projections with CI | `analytics_engine.py` |
| `recommendations` | Data-driven action items | Conditional logic on metrics |

Let's assemble and generate the complete PDF.

In [ ]:
# Build KPI cards for the report table
# Define KPI definitions matching the dashboard
KPI_DEFS = [
    {"id": "mrr", "name": "MRR", "col": "mrr", "fmt": "currency", "target": 500_000, "hib": True, "cat": "Revenue"},
    {"id": "arr", "name": "ARR", "col": "arr", "fmt": "currency", "target": 6_000_000, "hib": True, "cat": "Revenue"},
    {"id": "nrr", "name": "NRR", "col": "nrr", "fmt": "pct", "target": 1.10, "hib": True, "cat": "Revenue"},
    {"id": "logo_churn", "name": "Logo Churn", "col": "logo_churn_rate", "fmt": "pct", "target": 0.03, "hib": False, "cat": "Retention"},
    {"id": "nps", "name": "NPS", "col": "nps", "fmt": "int", "target": 50, "hib": True, "cat": "Satisfaction"},
    {"id": "ltv_cac", "name": "LTV:CAC", "col": "ltv_cac_ratio", "fmt": "ratio", "target": 3.0, "hib": True, "cat": "Efficiency"},
    {"id": "cac", "name": "CAC", "col": "cac", "fmt": "currency", "target": 8000, "hib": False, "cat": "Efficiency"},
    {"id": "rule_of_40", "name": "Rule of 40", "col": "rule_of_40", "fmt": "pct_pts", "target": 40, "hib": True, "cat": "Health"},
]

def fmt_value(val, fmt):
    if fmt == "currency":
        return f"${val:,.0f}"
    elif fmt == "pct":
        return f"{val * 100:.1f}%" if val <= 2 else f"{val:.1f}%"
    elif fmt == "int":
        return f"{val:.0f}"
    elif fmt == "ratio":
        return f"{val:.1f}x"
    elif fmt == "pct_pts":
        return f"{val:.1f}%"
    return str(val)

kpi_cards = []
for defn in KPI_DEFS:
    col = defn["col"]
    if col not in monthly.columns:
        continue
    value = float(last_row[col])
    prev = float(prev_row[col])
    mom = (value - prev) / prev if prev != 0 else 0.0

    # YoY (if 13+ months available)
    if len(monthly) >= 13:
        yoy_val = float(monthly[col].iloc[-13])
        yoy = (value - yoy_val) / yoy_val if yoy_val != 0 else 0.0
    else:
        yoy = 0.0

    tl = traffic_light_status(value, defn["target"], defn["hib"])

    kpi_cards.append({
        "id": defn["id"],
        "name": defn["name"],
        "value": round(value, 4),
        "formatted": fmt_value(value, defn["fmt"]),
        "change_mom": round(mom, 4),
        "change_yoy": round(yoy, 4),
        "traffic_light": tl,
        "target": defn["target"],
        "category": defn["cat"],
    })

# Display KPI cards
print(f"{'KPI':15s} {'Value':>12s} {'MoM':>8s} {'YoY':>8s} {'Status':>8s}")
print("-" * 55)
for kpi in kpi_cards:
    print(f"{kpi['name']:15s} {kpi['formatted']:>12s} {kpi['change_mom']*100:>+7.1f}% {kpi['change_yoy']*100:>+7.1f}% {kpi['traffic_light']:>8s}")

In [ ]:
# Assemble the complete data payload for the report generator
report_data = {
    "metrics": metrics,
    "kpis": kpi_cards,
    "waterfall": waterfall,
    "anomalies": all_anomalies,
    "forecast": forecast_data,
    "customer_data": metrics,
    "period": period,
}

# Generate the PDF -- English version
import time
start = time.time()
pdf_bytes_en = generate_report(report_data, lang="en")
elapsed_en = time.time() - start

# Generate the PDF -- Spanish version
start = time.time()
pdf_bytes_es = generate_report(report_data, lang="es")
elapsed_es = time.time() - start

# Save locally for inspection
os.makedirs("../reports", exist_ok=True)
en_path = f"../reports/kpi_report_{period['end']}_en.pdf"
es_path = f"../reports/kpi_report_{period['end']}_es.pdf"

with open(en_path, "wb") as f:
    f.write(pdf_bytes_en)
with open(es_path, "wb") as f:
    f.write(pdf_bytes_es)

print(f"PDF Generation Results")
print(f"=" * 50)
print(f"  English report: {len(pdf_bytes_en):,} bytes ({elapsed_en:.2f}s)")
print(f"  Spanish report: {len(pdf_bytes_es):,} bytes ({elapsed_es:.2f}s)")
print(f"\n  Saved to:")
print(f"    {en_path}")
print(f"    {es_path}")
print(f"\n  Sections included: {len(report_data['kpis'])} KPIs, "
      f"{len(report_data['anomalies'])} anomalies, "
      f"{len(report_data['forecast'])} forecast metrics")

## 7. Pipeline Visualization: What Goes Into the Report

Let's visualize the complete data flow and the key metrics that feed each section of the report.

In [ ]:
# Visualize the traffic light distribution across KPIs
tl_counts = {"green": 0, "yellow": 0, "red": 0}
for kpi in kpi_cards:
    tl_counts[kpi["traffic_light"]] = tl_counts.get(kpi["traffic_light"], 0) + 1

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("KPI Traffic Light Distribution", "Health Score Components"),
    specs=[[{"type": "pie"}, {"type": "bar"}]],
)

# Pie chart: traffic light distribution
fig.add_trace(go.Pie(
    labels=["Green", "Yellow", "Red"],
    values=[tl_counts.get("green", 0), tl_counts.get("yellow", 0), tl_counts.get("red", 0)],
    marker=dict(colors=[COLORS["emerald"], COLORS["amber"], COLORS["rose"]]),
    textinfo="label+value",
    hole=0.4,
), row=1, col=1)

# Bar chart: health score components
health_components = {
    "MRR Growth": min(100, max(0, (metrics.get("mrr_growth_rate", 0) - 0) / 0.05 * 100)),
    "NRR": min(100, max(0, (metrics.get("nrr", 1.0) - 0.95) / 0.20 * 100)),
    "Churn (inv.)": min(100, max(0, (0.05 - metrics.get("logo_churn_rate", 0.03)) / 0.03 * 100)),
    "NPS": min(100, max(0, (metrics.get("nps", 45) - 25) / 30 * 100)),
    "Rule of 40": min(100, max(0, (metrics.get("rule_of_40", 30) - 15) / 25 * 100)),
    "LTV:CAC": min(100, max(0, (metrics.get("ltv_cac_ratio", 3.0) - 1.5) / 2.5 * 100)),
}

bar_colors = [COLORS["cyan"], COLORS["violet"], COLORS["emerald"],
              COLORS["amber"], COLORS["rose"], COLORS["cyan"]]

fig.add_trace(go.Bar(
    x=list(health_components.keys()),
    y=list(health_components.values()),
    marker_color=bar_colors,
    text=[f"{v:.0f}" for v in health_components.values()],
    textposition="outside",
), row=1, col=2)

fig.update_yaxes(range=[0, 110], title_text="Score (0-100)", row=1, col=2)
fig.update_layout(
    height=400, template="plotly_white", showlegend=False,
    title="Report Data: KPI Status and Health Score Breakdown",
    font=dict(family="Inter, sans-serif"),
)
fig.show()

print(f"\nOverall Health Score: {metrics['health_score']:.1f} / 100")
print(f"Traffic Lights: {tl_counts['green']} green, {tl_counts['yellow']} yellow, {tl_counts['red']} red")

## 8. Section-Selective Generation

One powerful feature of the pipeline is the ability to generate reports with only specific sections. This is useful for:
- **Quick updates**: Generate only the executive summary and KPI table for a daily standup
- **Deep dives**: Generate only the anomaly alerts and forecast sections for a data review
- **Board packages**: Generate the full report for quarterly board meetings

Let's demonstrate generating a minimal "quick update" report.

In [ ]:
# Generate section-selective reports
report_configs = {
    "Quick Update (summary + KPIs)": ["cover", "executive_summary", "kpi_dashboard"],
    "Deep Dive (anomalies + forecast)": ["cover", "anomaly_alerts", "forecast", "recommendations"],
    "Full Board Report (all sections)": None,  # None = all sections
}

print("Section-Selective Report Generation")
print("=" * 60)

for name, sections in report_configs.items():
    start = time.time()
    pdf_bytes = generate_report(report_data, lang="en", sections=sections)
    elapsed = time.time() - start

    section_count = len(sections) if sections else 8
    print(f"\n  {name}:")
    print(f"    Sections: {section_count}")
    print(f"    Size:     {len(pdf_bytes):,} bytes")
    print(f"    Time:     {elapsed:.3f}s")

    if sections:
        print(f"    Included: {', '.join(sections)}")

## Summary

### Pipeline Architecture

The automated executive report generation pipeline consists of 6 stages:

1. **Data Loading** -- Read parquet files with 24 months of SaaS metrics
2. **KPI Computation** -- Derive health scores, traffic lights, MoM/YoY changes
3. **Anomaly Detection** -- Z-score analysis flagging unusual values
4. **Forecasting** -- Holt-Winters exponential smoothing with 95% confidence intervals
5. **NLG Commentary** -- Template-based bilingual narrative generation
6. **PDF Assembly** -- fpdf2-based document with charts, tables, and narrative

### Key Design Decisions

| Decision | Choice | Rationale |
|----------|--------|-----------|
| PDF library | fpdf2 | Pure Python, no system deps, fast |
| NLG approach | Template-based | Deterministic, auditable, bilingual |
| Chart rendering | plotly -> PNG (kaleido) | Same library as dashboard |
| Forecasting | Holt-Winters | Works with small samples (24 months) |
| Anomaly detection | Z-score (z=2.0) | Simple, interpretable for executives |

### Production Usage

In production, the pipeline runs via the FastAPI endpoint `GET /report/generate`:
- Query parameters: `segment`, `start_month`, `end_month`, `lang`, `sections`
- Returns: Streaming PDF download
- Latency: <1 second for full report (without chart rendering)

### Extending the Pipeline

To add a new section to the report:
1. Add a commentary function in `commentary.py`
2. Add section rendering logic in `report_generator.py`
3. Include the section name in the `all_sections` list
4. The FastAPI endpoint automatically picks it up

---
*Report generator source: `backend/kpi_backend/report_generator.py` | Commentary: `backend/kpi_backend/commentary.py`*